# NLP Fundamentals with spaCy

Explores core NLP concepts using spaCy: stop word removal, lemmatization, and Named Entity Recognition (NER) on real text.

In [1]:
!pip install spacy

!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 72.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import spacy

nlp = spacy.load('en_core_web_sm')

# Inspect spaCy's built-in stop word list
print(f'spaCy has {len(nlp.Defaults.stop_words)} English stop words')
print('Sample:', sorted(list(nlp.Defaults.stop_words))[:12])

# Remove stop words from a sentence
sentence = 'The quick brown fox jumps over the lazy dog near the river'
doc = nlp(sentence)

# Keep only words that are: alphabetic, not a stop word
content_words = [token.text for token in doc
                 if not token.is_stop and token.is_alpha]

print(f'Original ({len(doc)} tokens): {[t.text for t in doc]}')
print(f'Filtered ({len(content_words)} tokens): {content_words}')

# Customise the stop list - add domain-specific word to the stop list
nlp.vocab['data'].is_stop = True

# preserve negation words — removing 'not' flips sentiment!
# spaCy keeps 'not', 'no', 'never' etc. in the stop list by default.
# Be careful not to remove them for sentiment tasks.
safe_check = ['not', 'no', 'never', 'nor', 'neither']
for w in safe_check:
    print(f"  '{w}' is_stop = {nlp.vocab[w].is_stop}")

spaCy has 326 English stop words
Sample: ["'d", "'ll", "'m", "'re", "'s", "'ve", 'a', 'about', 'above', 'across', 'after', 'afterwards']
Original (12 tokens): ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', 'near', 'the', 'river']
Filtered (8 tokens): ['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog', 'near', 'river']
  'not' is_stop = True
  'no' is_stop = True
  'never' is_stop = True
  'nor' is_stop = True
  'neither' is_stop = True


In [3]:
import spacy

nlp = spacy.load('en_core_web_sm')

# Lemmatization in action
sentence = 'The geese were running faster than the mice had ever studied'
doc = nlp(sentence)

print(f'{'Token':<14} {'Lemma':<14} {'POS'}')
print('-' * 40)
for token in doc:
    if not token.is_punct and not token.is_space:
        print(f'{token.text:<14} {token.lemma_:<14} {token.pos_}')

Token          Lemma          POS
----------------------------------------
The            the            DET
geese          goose          NOUN
were           be             AUX
running        run            VERB
faster         fast           ADV
than           than           SCONJ
the            the            DET
mice           mouse          NOUN
had            have           AUX
ever           ever           ADV
studied        study          VERB


In [4]:
import spacy
from collections import defaultdict

nlp = spacy.load('en_core_web_sm')

text = (
    'Apple Inc. CEO Tim Cook announced on Tuesday that the company '
    'will invest $10 billion in manufacturing across the United States. '
    'Apple shares rose 3.5 percent on the Nasdaq.'
)
doc = nlp(text)

# Print every entity with its type
print(f"{'Entity':<28} {'Label':<12} {'Description'}")
print('-' * 65)
for ent in doc.ents:
    desc = spacy.explain(ent.label_) or ''
    print(f'{ent.text:<28} {ent.label_:<12} {desc}')

# Group by type
groups = defaultdict(set)
for ent in doc.ents:
    groups[ent.label_].add(ent.text)

print('\nGrouped by entity type:')
for label, entities in sorted(groups.items()):
    print(f'  {label:<12}: {", ".join(sorted(entities))}')

# Add custom entity patterns
# The base model might miss domain-specific entities.
# Use EntityRuler to add your own patterns before the NER component.
from spacy.pipeline import EntityRuler
ruler = nlp.add_pipe('entity_ruler', before='ner')
ruler.add_patterns([
    {'label': 'AI_MODEL', 'pattern': [{'TEXT': {'IN': ['BERT','GPT-4','T5','LLaMA']}}]},
    {'label': 'COUNTY',   'pattern': [{'LOWER': 'nairobi'}]},
])
doc2 = nlp('BERT and GPT-4 are widely used in Nairobi for AI research.')
for ent in doc2.ents:
    print(f'  Custom: {ent.text} -> {ent.label_}')

Entity                       Label        Description
-----------------------------------------------------------------
Apple Inc.                   ORG          Companies, agencies, institutions, etc.
Tim Cook                     PERSON       People, including fictional
Tuesday                      DATE         Absolute or relative dates or periods
$10 billion                  MONEY        Monetary values, including unit
the United States            GPE          Countries, cities, states
Apple                        ORG          Companies, agencies, institutions, etc.
3.5 percent                  PERCENT      Percentage, including "%"
Nasdaq                       ORG          Companies, agencies, institutions, etc.

Grouped by entity type:
  DATE        : Tuesday
  GPE         : the United States
  MONEY       : $10 billion
  ORG         : Apple, Apple Inc., Nasdaq
  PERCENT     : 3.5 percent
  PERSON      : Tim Cook
  Custom: BERT -> AI_MODEL
  Custom: GPT-4 -> AI_MODEL
  Custom: Nai

In [5]:
import spacy
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

nlp = spacy.load('en_core_web_sm')

# Custom tokenizer: use spaCy's lemmas as tokens
# This combines BoW with spaCy's linguistic processing.
# We lemmatize and remove stop words before vectorizing.
def spacy_tokenizer(text: str) -> list[str]:
    doc = nlp(text.lower())
    return [token.lemma_ for token in doc
            if not token.is_stop and not token.is_punct
            and token.is_alpha and len(token.text) > 2]

corpus = [
    'The cat sat on the mat',
    'The dog sat on the log',
    'The cat and the dog are friends',
]

# Use spaCy tokenizer inside sklearn's CountVectorizer
vectorizer = CountVectorizer(tokenizer=spacy_tokenizer)
X = vectorizer.fit_transform(corpus)

df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
df.index = [f'Doc {i+1}' for i in range(len(corpus))]
print(df)

       cat  dog  friend  log  mat  sit
Doc 1    1    0       0    0    1    1
Doc 2    0    1       0    1    0    1
Doc 3    1    1       1    0    0    0


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
